# FocusRoom — 02: Model Training
**Goal:** Train the Eye State CNN interactively and monitor it epoch-by-epoch.

This notebook is a **visual companion** to `train.py`. It does the same training but lets you:
- Watch loss/accuracy curves update live
- Inspect what the model has learned at any point
- Experiment with hyperparameters before committing to a full run

For a long overnight run, use `train.py` in the terminal — it handles crashes and resumes automatically.

**Prerequisites:** Run `01_data_exploration.ipynb` and `scripts/prepare_dataset.py` first.

In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard

# Add parent directory to path so we can import from train.py
sys.path.insert(0, os.path.abspath('..'))

# Config — keep in sync with train.py
CONFIG = {
    'data_dir':          '../data/processed',
    'checkpoint_dir':    '../checkpoints',
    'best_checkpoint':   '../checkpoints/best_model.keras',
    'latest_checkpoint': '../checkpoints/latest_epoch.keras',
    'img_size':          (48, 48),
    'num_classes':       3,
    'class_names':       ['focused', 'distracted', 'closed'],
    'batch_size':        32,
    'epochs':            60,
    'initial_lr':        1e-3,
    'dropout_rate':      0.4,
    'l2_lambda':         1e-4,
    'seed':              42,
}

tf.random.set_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✓ GPU: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('⚠ No GPU — training on CPU. Use Colab GPU runtime for speed.')

print(f'TensorFlow version: {tf.__version__}')

## 1. Load Data

In [ ]:
CLASSES = CONFIG['class_names']
IMG_H, IMG_W = CONFIG['img_size']

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.6, 1.4],
    fill_mode='nearest',
)
val_datagen = ImageDataGenerator(rescale=1.0/255.0)

common = dict(
    target_size=CONFIG['img_size'],
    color_mode='grayscale',
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    classes=CLASSES,
    seed=CONFIG['seed'],
)

train_gen = train_datagen.flow_from_directory(os.path.join(CONFIG['data_dir'], 'train'), shuffle=True, **common)
val_gen   = val_datagen.flow_from_directory(os.path.join(CONFIG['data_dir'], 'val'),   shuffle=False, **common)

# Class weights
class_counts = np.array([
    len(os.listdir(os.path.join(CONFIG['data_dir'], 'train', cls)))
    for cls in CLASSES
])
total = class_counts.sum()
class_weights = {i: total / (len(CLASSES) * c) for i, c in enumerate(class_counts)}

print('\nDataset loaded:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:12s}: {class_counts[i]:,} train images | weight = {class_weights[i]:.3f}')
print(f'  Val samples : {val_gen.samples:,}')

## 2. Build Model

In [ ]:
reg = regularizers.l2(CONFIG['l2_lambda'])
inputs = keras.Input(shape=(IMG_H, IMG_W, 1), name='eye_crop')

# Block 1
x = layers.Conv2D(32, (3,3), padding='same', kernel_regularizer=reg)(inputs)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Conv2D(32, (3,3), padding='same', kernel_regularizer=reg)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2,2))(x)  # → 24x24

# Block 2
x = layers.Conv2D(64, (3,3), padding='same', kernel_regularizer=reg)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Conv2D(64, (3,3), padding='same', kernel_regularizer=reg)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling2D((2,2))(x)  # → 12x12

# Block 3
x = layers.Conv2D(128, (3,3), padding='same', kernel_regularizer=reg)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Conv2D(128, (3,3), padding='same', kernel_regularizer=reg)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.GlobalAveragePooling2D()(x)  # → 128

# Head
x = layers.Dense(256, kernel_regularizer=reg)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.Dropout(CONFIG['dropout_rate'])(x)
outputs = layers.Dense(CONFIG['num_classes'], activation='softmax')(x)

model = keras.Model(inputs, outputs, name='eye_state_cnn')
model.compile(
    optimizer=keras.optimizers.Adam(CONFIG['initial_lr']),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')],
)
model.summary()

## 3. Train

In [ ]:
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

callbacks = [
    ModelCheckpoint(
        CONFIG['best_checkpoint'], monitor='val_accuracy',
        save_best_only=True, mode='max', verbose=1
    ),
    ModelCheckpoint(
        CONFIG['latest_checkpoint'], monitor='val_loss',
        save_best_only=False, verbose=0  # resume checkpoint
    ),
    EarlyStopping(monitor='val_accuracy', patience=10, min_delta=0.001,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_gen,
    epochs=CONFIG['epochs'],
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks,
)

# Save history for plotting
with open('../checkpoints/training_history.json', 'w') as f:
    json.dump(history.history, f, indent=2)
print('History saved to checkpoints/training_history.json')

## 4. Training Curves

In [ ]:
hist = history.history
epochs_ran = range(1, len(hist['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Training Curves — Eye State CNN', fontsize=14, fontweight='bold')

# Accuracy
axes[0].plot(epochs_ran, hist['accuracy'],     '#2D6A4F', linewidth=2, label='Train Accuracy')
axes[0].plot(epochs_ran, hist['val_accuracy'], '#E63946', linewidth=2, linestyle='--', label='Val Accuracy')
axes[0].axhline(0.88, color='#E76F51', linestyle=':', linewidth=1.5, label='Target (88%)')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)
best_val = max(hist['val_accuracy'])
best_ep  = hist['val_accuracy'].index(best_val) + 1
axes[0].annotate(f'Best: {best_val:.4f}\n(epoch {best_ep})',
                 xy=(best_ep, best_val), xytext=(best_ep+2, best_val-0.05),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

# Loss
axes[1].plot(epochs_ran, hist['loss'],     '#1A1A2E', linewidth=2, label='Train Loss')
axes[1].plot(epochs_ran, hist['val_loss'], '#E63946', linewidth=2, linestyle='--', label='Val Loss')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../checkpoints/training_curves.png', dpi=120)
plt.show()
print(f'Best val_accuracy: {best_val:.4f} at epoch {best_ep}')
print(f'Target met (≥88%): {"✓ YES" if best_val >= 0.88 else "✗ NO — need more data or tuning"}')
print('Saved → checkpoints/training_curves.png')

## 5. Quick Overfitting Check

In [ ]:
# Overfitting = train acc much higher than val acc
final_train = hist['accuracy'][-1]
final_val   = hist['val_accuracy'][-1]
gap = final_train - final_val

print('Overfitting analysis:')
print(f'  Final train accuracy : {final_train:.4f}')
print(f'  Final val accuracy   : {final_val:.4f}')
print(f'  Gap (train - val)    : {gap:.4f}')
print()
if gap < 0.05:
    print('  ✓ Gap < 5% — good generalisation. Model is NOT overfitting.')
elif gap < 0.10:
    print('  ⚠ Gap 5-10% — mild overfitting. Consider more augmentation or stronger dropout.')
else:
    print('  ✗ Gap > 10% — significant overfitting.')
    print('  Fixes: increase dropout_rate, increase l2_lambda, add more training data.')

## 6. Next Steps

In [ ]:
print('═'*52)
print('  TRAINING COMPLETE')
print('═'*52)
print(f'  Best model → checkpoints/best_model.keras')
print(f'  Best val acc: {best_val:.4f}  (target ≥ 0.88)')
print()
print('  Next steps:')
print('    1. Open 03_evaluation.ipynb for detailed metrics')
print('    2. Run: python ../evaluate.py')
print('    3. Run: python ../export.py')
print('═'*52)